MODEL (baseline xgboost on all features)

In [ ]:
#pip install xgboost scikit-learn  (for macbook also: brew install libomp)
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from cohort import getCohort
from features import getFeatures, getGroupColumns

testSize = 0.2
randomState = 42
#above this probability we predict failure
threshold = 0.5

FEATURES

In [ ]:
features = getFeatures(getCohort())
groupColumns = getGroupColumns(features)
for group, columns in groupColumns.items():
    print(group, len(columns))
print(f"extubations: {len(features)}, failure rate: {features['failure'].mean():.3f}")

SPLIT on person_id 

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=testSize, random_state=randomState)
trainIndex, testIndex = next(splitter.split(features, groups=features["person_id"]))
train = features.iloc[trainIndex]
test = features.iloc[testIndex]
print(f"train: {len(train)} failure rate {train['failure'].mean():.3f}, test: {len(test)} failure rate {test['failure'].mean():.3f}")

MODEL. TUNE LATER!!

In [ ]:
def getModel():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        enable_categorical=True,
        tree_method="hist",
        eval_metric="logloss",
        random_state=randomState,
    )

BASELINE all features together

In [ ]:
columns = groupColumns["all"]
model = getModel()
model.fit(train[columns], train["failure"].astype(int))
testProbability = model.predict_proba(test[columns])[:, 1]


print(f"auroc: {roc_auc_score(test['failure'], testProbability):.3f}")
print(f"auprc: {average_precision_score(test['failure'], testProbability):.3f} (random {test['failure'].mean():.3f})")

#always predicting success already gives about 0.80 be aware
testPrediction = (testProbability >= threshold).astype(int)
print(f"accuracy: {accuracy_score(test['failure'], testPrediction):.3f} (always success {1 - test['failure'].mean():.3f})")
print(f"precision: {precision_score(test['failure'], testPrediction, zero_division=0):.3f}")
print(f"recall: {recall_score(test['failure'], testPrediction):.3f}")
print(f"f1: {f1_score(test['failure'], testPrediction):.3f}")
trueNegative, falsePositive, falseNegative, truePositive = confusion_matrix(test['failure'], testPrediction).ravel()
print(f"specificity: {trueNegative / (trueNegative + falsePositive):.3f}")
print(f"confusion matrix: tn {trueNegative}, fp {falsePositive}, fn {falseNegative}, tp {truePositive}")

In [ ]:
#most important features
importance = pd.Series(model.feature_importances_, index=columns).sort_values(ascending=False)
print(importance.head(20).to_string())